### Extract Faces

In [1]:
import os
import tensorflow as tf
import cv2
import numpy as np
from pathlib import Path
from mtcnn import MTCNN
from tqdm.auto import tqdm      # auto: pakai notebook jika tersedia, else console
from utils import load_annotations, read_ocean_data
import matplotlib.pyplot as plt


def extract_face(image, image_size=(112, 112)):
    """
    Deteksi dan crop wajah dari satu frame menggunakan MTCNN.
    Mengembalikan gambar wajah yang sudah di-resize, atau None jika tidak terdeteksi.
    """
    detector = MTCNN()
    faces = detector.detect_faces(image)
    if len(faces) > 0:
        x, y, width, height = faces[0]['box']
        x, y = max(0, x), max(0, y)
        x2, y2 = x + width, y + height
        face = image[y:y2, x:x2]
        if face.size > 0:
            face = cv2.resize(face, image_size)
            return face
    return None


def load_faces(video_dir, save_dir, annotation, image_size=(112, 112), num_images=10):
    """
    Memproses semua video di video_dir:
    - Ekstrak 10 frame wajah per video menggunakan MTCNN
    - Ambil label OCEAN yang benar dari annotation
    - Tampilkan progress bar 1-100%
    - Kembalikan sebagai tf.data.Dataset
    """
    all_video_files = [
        f for f in os.listdir(video_dir)
        if os.path.isfile(os.path.join(video_dir, f))
        and f.lower().endswith('.mp4')
    ]
    total = len(all_video_files)

    X_all = []   # kumpulan frame wajah
    y_all = []   # kumpulan label OCEAN
    skipped = 0

    # Progress bar visual di Jupyter
    pbar = tqdm(
        all_video_files,
        total=total,
        desc=f'Ekstraksi wajah ({save_dir.split("/")[-1]})',
        unit='video',
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} video [{elapsed}<{remaining}, {rate_fmt}]'
    )

    for video_name in pbar:
        video_path = os.path.normpath(os.path.join(video_dir, video_name))
        video_stem = Path(video_name).stem

        # Ambil label OCEAN yang benar
        label = read_ocean_data(video_stem, annotation)
        if label is None:
            skipped += 1
            pbar.set_postfix({'skip': skipped, 'ok': len(X_all)})
            continue

        # Buat folder simpan wajah
        save_path = os.path.normpath(os.path.join(save_dir, video_stem))
        os.makedirs(save_path, exist_ok=True)

        # Buka video dan ekstrak frame
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        interval = max(frame_count // num_images, 1)
        face_images = []
        last_valid_face = np.zeros((image_size[0], image_size[1], 3), dtype=np.float32)

        for i in range(num_images):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
            ret, frame = cap.read()
            if not ret:
                face_images.append(last_valid_face)
                continue

            face = extract_face(frame, image_size)
            if face is not None:
                face_np = np.asarray(face, dtype=np.float32)
                face_images.append(face_np)
                last_valid_face = face_np
                cv2.imwrite(os.path.join(save_path, f'face_{i+1}.jpg'), face)
            else:
                face_images.append(last_valid_face)

        cap.release()

        # Pastikan tepat num_images frame
        while len(face_images) < num_images:
            face_images.append(last_valid_face)

        X_all.append(np.array(face_images[:num_images], dtype=np.float32))
        y_all.append(label)
        pbar.set_postfix({'skip': skipped, 'ok': len(X_all)})

    pbar.close()
    print(f'\nSelesai: {len(X_all)} video berhasil | {skipped} di-skip (tidak ada anotasi)')

    # Buat tf.data.Dataset dari list yang sudah terkumpul
    X_np = np.array(X_all, dtype=np.float32)
    y_np = np.array(y_all, dtype=np.float32)

    def generator():
        for x, y in zip(X_np, y_np):
            yield x, y

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(num_images, image_size[0], image_size[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(5,), dtype=tf.float32)
        )
    )
    return dataset


# ── Proses semua subset ────────────────────────────────────────────────────────
subsets = ['train', 'val', 'test']

annotation_train, annotation_val, annotation_test = load_annotations()
annotations = {
    'train': annotation_train,
    'val'  : annotation_val,
    'test' : annotation_test,
}

for subset in subsets:
    print(f'\n{"="*50}')
    print(f'Subset: {subset.upper()}')
    print('='*50)

    video_dir = os.path.normpath(f'dataset/dataset_100/{subset}')
    save_dir  = os.path.normpath(f'dataset/dataset_100/videofacecrop_100_{subset}')
    os.makedirs(save_dir, exist_ok=True)

    dataset = load_faces(video_dir, save_dir, annotations[subset]).batch(8)

    # Simpan dataset
    save_path_ds = f'data/videoface_100/{subset}_ds'
    tf.data.experimental.save(dataset, save_path_ds)

    # Verifikasi label
    sample = [(y.numpy()) for _, y in dataset.take(1)]
    if sample:
        print(f'Contoh label [{subset}]: {sample[0][0].round(4)}')
        status = 'OK - label benar' if sample[0].sum() > 0 else '[PERINGATAN] Label masih nol!'
        print(f'Status label : {status}')
    print(f'Dataset disimpan ke: {save_path_ds}')

print('\nSemua dataset selesai di-generate!')


c:\Users\ALFIAN\anaconda3\envs\env_insightface\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ALFIAN\anaconda3\envs\env_insightface\lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Update annotation path !

Subset: TRAIN


Ekstraksi wajah (dataset\dataset_100\videofacecrop_100_train):   0%|          | 0/60 video [00:00<?, ?video/s]

Ekstraksi wajah (dataset\dataset_100\videofacecrop_100_train): 100%|██████████| 60/60 video [04:40<00:00,  4.68s/video]


Selesai: 60 video berhasil | 0 di-skip (tidak ada anotasi)
Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


Contoh label [train]: [0.8222 0.534  0.6075 0.7033 0.6146]
Status label : OK - label benar
Dataset disimpan ke: data/videoface_100/train_ds

Subset: VAL


Ekstraksi wajah (dataset\dataset_100\videofacecrop_100_val): 100%|██████████| 20/20 video [01:33<00:00,  4.66s/video]



Selesai: 20 video berhasil | 0 di-skip (tidak ada anotasi)
Contoh label [val]: [0.6778 0.5243 0.6075 0.7473 0.4896]
Status label : OK - label benar
Dataset disimpan ke: data/videoface_100/val_ds

Subset: TEST


Ekstraksi wajah (dataset\dataset_100\videofacecrop_100_test): 100%|██████████| 20/20 video [01:35<00:00,  4.77s/video]


Selesai: 20 video berhasil | 0 di-skip (tidak ada anotasi)
Contoh label [test]: [0.7556 0.3495 0.4953 0.5385 0.4583]
Status label : OK - label benar
Dataset disimpan ke: data/videoface_100/test_ds

Semua dataset selesai di-generate!
